# Module 5: Train a Debugging Agent with GRPO

Fine-tune Qwen3-1.7B to automatically fix broken ML training scripts using GRPO + the ML Pipeline Debugger environment.

**Time:** ~2 hours (training) · **Difficulty:** Advanced · **GPU:** A100 required

In [ ]:
!pip install -Uq 'trl>=0.17.0' openenv-core transformers torch numpy vllm datasets
import sys, os
sys.path.insert(0, os.path.abspath('ml-pipeline-debugger'))
print('Setup complete!')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Connect to the Environment

Point at your deployed HF Space (or localhost for local training).

In [ ]:
from client import MLPipelineDebuggerEnv
from models import MLDebugAction

# Replace with your deployed Space URL
ENV_URL = 'https://your-username-ml-pipeline-debugger.hf.space'
# ENV_URL = 'http://localhost:8000'  # for local training

# Verify connection
with MLPipelineDebuggerEnv(base_url=ENV_URL).sync() as check:
    result = check.reset()
    print(f'Connected to: {ENV_URL}')
    print(f'Task: {result.observation.task_id}, Difficulty: {result.observation.difficulty}')
    print(f'Bug: {result.observation.bug_type}')

# Create persistent training connection
env = MLPipelineDebuggerEnv(base_url=ENV_URL)
sync_env = env.sync()
sync_env.connect()
print('Persistent training connection established.')

## 2. Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f'Model: {model_name}')

## 3. System Prompt — Expert ML Debugger

In [ ]:
SYSTEM_PROMPT = """
You are an expert ML engineer specialised in debugging broken PyTorch training scripts.

You receive:
- SYMPTOM: what is going wrong (exploding loss, NaN, flat loss curve)
- LOSS CURVE: first N loss values from running the broken script
- BUG TYPE: category of the injected fault
- BROKEN SCRIPT: the complete Python script that needs fixing

Your task: return ONLY the complete fixed Python script.

CRITICAL RULES:
1. Return ONLY the complete Python script — no markdown, no explanation
2. Every epoch MUST print exactly: loss:X.XXXXXX
3. Minimal fix — change only what is broken
4. The script must be runnable as-is with python -c

Bug categories you may see:
- lr_too_high / lr_zero: fix the optimizer lr parameter
- weight_decay_too_high: fix weight_decay
- momentum_negative: fix momentum sign
- epochs_zero: fix range() argument
- div_by_zero_std: add guard before dividing by std
- log_of_zero: clip values before log()
- sqrt_negative: use abs() before sqrt()
- inf_from_scale: remove extreme scaling
- missing_fillna: fill NaN before tensor cast
- zero_weight_init: remove constant_ init
- frozen_layer: set requires_grad=True
- missing_optimizer_step: add optimizer.step()
- wrong_activation: replace with torch.relu
"""

## 4. Reward Functions

The grader already returns a composite score 0.0–1.0. We expose it as a
single reward function to `GRPOTrainer`.

In [ ]:
def reward_total(completions, **kwargs):
    """Full grader score — 0.0 to 1.0 with partial credit."""
    rewards = kwargs.get('total_reward')
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_runs(completions, **kwargs):
    """+0.3 if the fixed script runs without crashing."""
    rewards = kwargs.get('runs_reward')
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_decreasing(completions, **kwargs):
    """+0.4 if the loss curve is decreasing."""
    rewards = kwargs.get('decreasing_reward')
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

print('Reward functions: total, runs, decreasing')

## 5. Rollout Function

One rollout = one debugging episode. The model reads the broken script and
generates a fix. The grader scores it.

In [ ]:
from trl.experimental.openenv import generate_rollout_completions

def rollout_once(trainer, sync_env, tokenizer, dataset_prompt, system_prompt):
    """One debugging episode: reset → generate fix → step → return scores."""
    result = sync_env.reset()
    obs = result.observation

    user_msg = f"""TASK: {obs.task_id} | DIFFICULTY: {obs.difficulty} | BUG: {obs.bug_type}

SYMPTOM:
{obs.task_description[:500]}

LOSS CURVE: {obs.loss_curve[:10]}

BROKEN SCRIPT:
{obs.broken_script}

Return the complete fixed Python script only."""

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=False, enable_thinking=False,
    )

    # Generate the fix
    rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
    fixed_code = rollout_outputs.get('text') or tokenizer.decode(
        rollout_outputs['completion_ids'], skip_special_tokens=True
    )
    # Strip markdown fences
    if '```' in fixed_code:
        fixed_code = '\n'.join(l for l in fixed_code.split('\n') if not l.startswith('```'))

    # Submit to grader
    step_result = sync_env.step(MLDebugAction(
        fixed_code=fixed_code,
        explanation='GRPO agent fix',
        task_id=obs.task_id,
    ))

    total_score = float(step_result.reward or 0.0)

    return {
        'prompt_ids':     rollout_outputs['prompt_ids'],
        'completion_ids': rollout_outputs['completion_ids'],
        'logprobs':       rollout_outputs['logprobs'],
        'total_reward':   total_score,
        'runs_reward':    min(total_score, 0.3),
        'decreasing_reward': max(0.0, total_score - 0.3),
    }


def rollout_func(prompts, trainer=None):
    episode_prompt_ids, episode_completion_ids, episode_logprobs = [], [], []
    total_rewards, runs_rewards, dec_rewards = [], [], []

    for prompt_text in prompts:
        ep = rollout_once(trainer, sync_env, tokenizer, prompt_text, SYSTEM_PROMPT)
        episode_prompt_ids.append(ep['prompt_ids'])
        episode_completion_ids.append(ep['completion_ids'])
        episode_logprobs.append(ep['logprobs'])
        total_rewards.append(ep['total_reward'])
        runs_rewards.append(ep['runs_reward'])
        dec_rewards.append(ep['decreasing_reward'])

    return {
        'prompt_ids': episode_prompt_ids,
        'completion_ids': episode_completion_ids,
        'logprobs': episode_logprobs,
        'total_reward': total_rewards,
        'runs_reward': runs_rewards,
        'decreasing_reward': dec_rewards,
    }

print('Rollout function defined.')

## 6. Dataset — One Prompt Per Episode

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    'prompt': ['Debug the broken ML training script.'] * 500
})
print(f'Dataset: {len(dataset)} episodes')

## 7. GRPO Config

In [ ]:
from trl import GRPOConfig

output_dir = 'ml-debugger-grpo-Qwen3-1.7B'

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=32,
    per_device_train_batch_size=1,
    warmup_steps=10,
    num_generations=4,          # 4 fixes per broken script
    max_completion_length=2048, # Fixes can be long
    max_prompt_length=3000,
    use_vllm=True,
    vllm_mode='colocate',
    vllm_gpu_memory_utilization=0.15,
    output_dir=output_dir,
    report_to='none',
    logging_steps=1,
    save_steps=50,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    push_to_hub=True,
)

print(f'Output directory: {output_dir}')
print(f'Generations per episode: {grpo_config.num_generations}')

## 8. Create Trainer and Train

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_runs, reward_decreasing],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)
print('Trainer created.')

In [ ]:
import torch
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name} — {round(gpu.total_memory/1e9,1)} GB')

In [ ]:
# Train (~2 hours on A100)
trainer_stats = trainer.train()
print(f'Training time: {round(trainer_stats.metrics["train_runtime"]/60, 1)} minutes')

In [ ]:
sync_env.close()
trainer.save_model(output_dir)
trainer.push_to_hub()
print(f'Model saved and pushed to Hub as {output_dir}')

## 9. Evaluate the Trained Model

In [ ]:
from transformers import AutoModelForCausalLM
import statistics

fine_tuned = AutoModelForCausalLM.from_pretrained(
    output_dir, torch_dtype='auto', device_map='auto'
)

def evaluate_model(model, tokenizer, env_url, episodes=15):
    results = []
    with MLPipelineDebuggerEnv(base_url=env_url).sync() as env:
        for ep in range(episodes):
            obs = env.reset().observation
            user_msg = f'BUG: {obs.bug_type}\nSYMPTOM: {obs.task_description[:200]}\n\nSCRIPT:\n{obs.broken_script}'
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_msg}]
            prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
            inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
            with torch.no_grad():
                output_ids = model.generate(**inputs, max_new_tokens=2000)
            fixed = tokenizer.decode(output_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
            result = env.step(MLDebugAction(fixed_code=fixed, explanation='eval', task_id=obs.task_id))
            results.append({'task': obs.task_id, 'difficulty': obs.difficulty, 'score': result.reward or 0.0})
            print(f'ep{ep+1:2d}: {obs.task_id} ({obs.difficulty}) bug={obs.bug_type} score={result.reward:.2f}')
    return results

results = evaluate_model(fine_tuned, tokenizer, ENV_URL)

print('\n=== Evaluation Summary ===')
for diff in ['easy', 'medium', 'hard']:
    scores = [r['score'] for r in results if r['difficulty'] == diff]
    if scores:
        print(f'{diff:6s}: mean={statistics.mean(scores):.3f} (n={len(scores)})')
all_scores = [r['score'] for r in results]
print(f'OVERALL: mean={statistics.mean(all_scores):.3f}')

## Summary

What happened:
1. Connected Qwen3-1.7B to the ML Pipeline Debugger via GRPO
2. Model generated 4 candidate fixes per broken script
3. Grader scored each fix 0.0–1.0 (deterministically)
4. GRPO updated toward higher-scoring fixes

The key insight: **the environment is the plug-in**. Swap it for any other
OpenEnv environment and the training pipeline stays identical.

**Task 3 (silent underfitting) remains hard** — the model must reason across
multiple files with no error message. This is the core research frontier.